# Types of Prompts

Source slide: `slides/prompt-engineering/02_Types_of_Prompts.md`

Each section below demonstrates one prompt type named in the slide using the real GitHub Copilot SDK.



## Prerequisites

- Install the real GitHub Copilot SDK: `pip install github-copilot-sdk`
- Sign in to GitHub Copilot in VS Code or GitHub Codespaces
- These examples use ambient auth; do not paste a PAT into the notebook

Each technique below has a real Copilot SDK failure run, a short failure test, and an improved run that shows the fix.


In [ ]:
from copilot import CopilotClient
from copilot.session import PermissionHandler

model = "gpt-4.1"

async def ask_copilot(
    prompt: str,
    *,
    system_message: str | None = None,
    model_name: str = model,
) -> str:
    """Run a single prompt through the real GitHub Copilot SDK using ambient auth."""
    async with CopilotClient() as client:
        session_kwargs = {
            "model": model_name,
            "on_permission_request": PermissionHandler.approve_all,
        }
        if system_message:
            session_kwargs["system_message"] = {
                "mode": "append",
                "content": system_message,
            }
        async with await client.create_session(**session_kwargs) as session:
            response = await session.send_and_wait(prompt)
            return response.data.content if response and response.data else ""

def show_block(title: str, content: str) -> None:
    print(title)
    print("=" * 80)
    print(content.strip())
    print()

async def run_prompt_pair(
    *,
    technique_name: str,
    failure_prompt: str,
    improved_prompt: str,
    failure_test: str,
    fix_explanation: str,
    system_message: str | None = None,
    config_note: str | None = None,
) -> None:
    show_block("📘 Technique", technique_name)
    if config_note:
        show_block("⚙️ Configuration note", config_note)
    show_block("❌ Failure prompt", failure_prompt)
    failure_result = await ask_copilot(
        failure_prompt,
        system_message=system_message,
    )
    show_block("❌ Failure result", failure_result)
    show_block("🧪 Failure test", failure_test)
    show_block("✅ Improved prompt", improved_prompt)
    improved_result = await ask_copilot(
        improved_prompt,
        system_message=system_message,
    )
    show_block("✅ Improved result", improved_result)
    show_block("🔧 Why this fixes it", fix_explanation)

print("✅ Setup complete - real GitHub Copilot SDK with ambient auth")
print(f"📦 Using model: {model}")


## Zero-shot

**Failure mode:** Zero-shot prompting means giving a direct instruction without examples. New prompt engineers often fail by assuming the model will infer the exact format, label set, or level of detail they wanted.

**Failure test:** The failure prompt asks for ticket triage but never names the allowed labels or response format, so the model can respond with prose instead of one clean label.

**Technique:** Keep the prompt example-free, but make the task, label set, and output format explicit.

**Improved example:** The improved prompt is still zero-shot, but it adds the allowed labels and tells the model to answer with only one label, which removes avoidable ambiguity.


In [ ]:
await run_prompt_pair(
    technique_name='Zero-shot',
    failure_prompt='Triage this support request: “Our production login flow is down for every customer.”',
    improved_prompt='Classify this support request as exactly one of P0, P1, or P2. Reply with only the label. Request: “Our production login flow is down for every customer.”',
    failure_test='The failure prompt asks for ticket triage but never names the allowed labels or response format, so the model can respond with prose instead of one clean label.',
    fix_explanation='The improved prompt is still zero-shot, but it adds the allowed labels and tells the model to answer with only one label, which removes avoidable ambiguity.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Few-shot

**Failure mode:** Few-shot prompting teaches the pattern with a small number of examples. Without examples, the model may understand the task but miss your desired mapping or style.

**Failure test:** The failure prompt asks for intent classification but never demonstrates how “request,” “question,” and “bug” should be applied, so borderline cases can drift.

**Technique:** Add one or two representative examples that show the exact mapping you want the model to learn.

**Improved example:** The improved prompt includes labeled examples before the new item, so the model has a concrete pattern to continue instead of guessing your taxonomy.


In [ ]:
await run_prompt_pair(
    technique_name='Few-shot',
    failure_prompt='Label this ticket as bug, question, or request: “Please add dark mode to the dashboard.”',
    improved_prompt='Label each ticket as bug, question, or request.\n\nTicket: “The password reset link returns 500.”\nLabel: bug\n\nTicket: “How do I export my billing history?”\nLabel: question\n\nTicket: “Please add dark mode to the dashboard.”\nLabel:',
    failure_test='The failure prompt asks for intent classification but never demonstrates how “request,” “question,” and “bug” should be applied, so borderline cases can drift.',
    fix_explanation='The improved prompt includes labeled examples before the new item, so the model has a concrete pattern to continue instead of guessing your taxonomy.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Multi-shot

**Failure mode:** Multi-shot prompting extends the same idea with several examples. It is useful when a task has edge cases or subtle distinctions that one example cannot teach well.

**Failure test:** The failure prompt asks for sentiment labeling on a nuanced review, but it supplies no calibration examples for mixed or borderline sentiment.

**Technique:** Provide multiple examples that include normal cases and edge cases so the model can infer the decision boundary.

**Improved example:** The improved prompt shows several labeled examples, including a mixed review, so the model has a better template for handling ambiguity.


In [ ]:
await run_prompt_pair(
    technique_name='Multi-shot',
    failure_prompt='Classify the sentiment of this review as positive, neutral, negative, or mixed: “The dashboard is powerful, but setup took far too long.”',
    improved_prompt='Classify each review as positive, neutral, negative, or mixed.\n\nReview: “The app is fast and intuitive.”\nLabel: positive\n\nReview: “The features are okay, but nothing stands out.”\nLabel: neutral\n\nReview: “The release broke our export jobs.”\nLabel: negative\n\nReview: “The dashboard is powerful, but setup took far too long.”\nLabel: mixed\n\nReview: “The dashboard is powerful, but setup took far too long.”\nLabel:',
    failure_test='The failure prompt asks for sentiment labeling on a nuanced review, but it supplies no calibration examples for mixed or borderline sentiment.',
    fix_explanation='The improved prompt shows several labeled examples, including a mixed review, so the model has a better template for handling ambiguity.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
